# SVM

In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset

from task6.utils.prepare_data import prepare_data

C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Szymon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
dataset = load_dataset("google-research-datasets/go_emotions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [3]:
df = pd.DataFrame(dataset["train"])
df.head()

,text,labels,id
0,My favourite food is anything I didn't have to...,[27],eebbqej
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj
3,To make her feel threatened,[14],ed7ypvh
4,Dirty Southern Wankers,[3],ed0bdzj


In [4]:
df, emotions = prepare_data(df, "text", "labels")
print(emotions)
df.head()

Detected dataset type: goemotions
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed
['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


,text,id,ekman_emotion,tokenized_text,lemmatized_text
0,My favourite food is anything I didn't have to...,eebbqej,6,"[favourite, food, nt, cook]","[favourite, food, not, cook]"
1,"Now if he does off himself, everyone will thin...",ed00q6i,6,"[think, s, having, laugh, screwing, people, in...","[think, s, have, laugh, screw, people, instead..."
2,WHY THE FUCK IS BAYLESS ISOING,eezlygj,0,"[fuck, bayless, isoing]","[fuck, bayless, isoe]"
3,To make her feel threatened,ed7ypvh,2,"[feel, threatened]","[feel, threaten]"
4,Dirty Southern Wankers,ed0bdzj,0,"[dirty, southern, wankers]","[dirty, southern, wanker]"


In [5]:
df["ekman_emotion"].value_counts()

ekman_emotion
3    16920
6    12823
0     5579
5     4160
4     2599
2      691
1      638
Name: count, dtype: int64

## Train SVM

In [6]:
df_test = pd.read_csv("../../data/transcript_spell_checked.csv")
df_test, emotions_test = prepare_data(df_test, "Translation", "emotion_final")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [7]:
df_test.head()

,Start Time,End Time,Sentence,Translation,Emotion_fine,Emotion_core,Intensity,sentence_length,emotion_coverage,predicted_emotion,Sentence_corrected,ekman_emotion,tokenized_text,lemmatized_text
0,00:00:00,00:00:07,Program zawiera treści nieodpowiednie dla widz...,The program contains content inappropriate for...,warning,fear,mild,14,1,disgust,Program zawiera treści nieodpowiednie dla widz...,1,"[program, contains, content, inappropriate, un...","[program, contain, content, inappropriate, und..."
1,00:00:07,00:00:09,Oglądasz na własną odpowiedzialność.,You watch at your own risk.,caution,fear,mild,4,1,fear,Oglądasz na własną odpowiedzialność.,2,"[watch, risk]","[watch, risk]"
2,00:00:10,00:00:13,Jedziemy do Piękowa!,We're going to Pięków!,excitement,happiness,moderate,3,1,happiness,Jedziemy do Piękowa!,3,"[going, pikw]","[go, pikw]"
3,00:00:14,00:00:17,"Krakowiacek jeden, a okoliców siedem.","One Krakowiacek, and seven surroundings.",pride,happiness,mild,5,1,neutral,"Krakowiacek jeden, a okoliców siedem.",3,"[krakowiacek, seven, surroundings]","[krakowiacek, seven, surrounding]"
4,00:00:19,00:00:22,"Tak go dźgnąłem w tą tarczę i widziałem, że si...",So I stabbed him in that shield and I saw that...,irritation,anger,moderate,11,1,anger,"Tak go dźgnąłem w tą tarczę i widziałem, że si...",0,"[stabbed, shield, saw, got, pissed]","[stab, shield, see, get, piss]"


In [8]:
from task6.utils.tfidf import create_unified_tfidf_matrices

df_processed, df_test_processed, vectorizer = create_unified_tfidf_matrices(
    train_df=df,
    test_df=df_test,
    column="lemmatized_text",
    max_features=10000
)


Fitting TF-IDF vectorizer on training data...
Transforming test data with same vectorizer...
Training features: 10000
Test features: 10000
✓ Feature dimensions match: True


In [9]:
X_train = np.array(df_processed["lemmatized_text_tfidf"].tolist())
y_train = df_processed["ekman_emotion"]
X_test = np.array(df_test_processed["lemmatized_text_tfidf"].tolist())
y_test = df_test_processed["ekman_emotion"]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (43410, 10000), y_train shape: (43410,)
X_test shape: (469, 10000), y_test shape: (469,)


In [10]:
import numpy as np
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import LinearSVC
import os

# ===== DIAGNOSE THE MEMORY ISSUE =====
print("System diagnostics:")
print(f"Available CPU cores: {os.cpu_count()}")
print(f"Training matrix: {X_train.shape[0]:,} × {X_train.shape[1]:,}")
print(f"Memory usage: {X_train.nbytes / 1e9:.2f} GB")
print(f"Expected CV memory: ~{X_train.nbytes * 5 / 1e9:.2f} GB")

System diagnostics:
Available CPU cores: 16
Training matrix: 43,410 × 10,000
Memory usage: 3.47 GB
Expected CV memory: ~17.36 GB


In [11]:
n_samples, n_features = X_train.shape
dual_choice = True if n_samples > n_features else False

In [15]:
param_grid = {
    "C": [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0],
    "max_iter": [5000, 10000, 15000, 20000, 25000],
    "class_weight": [None, "balanced"],
    "loss": ["hinge", "squared_hinge"],
    "penalty": ["l2"]  # L1 penalty often causes issues with LinearSVC
}

randomized_search = RandomizedSearchCV(
    LinearSVC(random_state=42, dual=dual_choice),
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2,
    n_iter=50,
    scoring='f1_macro',
    random_state=42,
    return_train_score=True
)

print("Starting randomized search (50 iterations)...")
randomized_search.fit(X_train, y_train)

print("\n" + "="*50)
print("RANDOMIZED SEARCH RESULTS")
print("="*50)
print(f"Best parameters: {randomized_search.best_params_}")
print(f"Best CV score: {randomized_search.best_score_:.4f}")
print(f"Best estimator: {randomized_search.best_estimator_}")

Starting randomized search (50 iterations)...
Fitting 3 folds for each of 50 candidates, totalling 150 fits

RANDOMIZED SEARCH RESULTS
Best parameters: {'penalty': 'l2', 'max_iter': 20000, 'loss': 'squared_hinge', 'class_weight': 'balanced', 'C': 0.01}
Best CV score: 0.4806
Best estimator: LinearSVC(C=0.01, class_weight='balanced', dual=True, max_iter=20000,
          random_state=42)


Most often chosen parameters:

RANDOMIZED SEARCH RESULTS

Best parameters: {'penalty': 'l2', 'max_iter': 10000, 'loss': 'squared_hinge', 'class_weight': 'balanced', 'C': 0.1}

In [20]:
param_grid = {
    'C': [0.02, 0.05, 0.1, 0.2, 0.5],
    'max_iter': [20000],
    'class_weight': ['balanced'],
    'loss': ['squared_hinge'],
    'penalty': ['l2']
}
grid_search = GridSearchCV(
    LinearSVC(random_state=42, dual=dual_choice),
    param_grid,
    cv=3,
    n_jobs=8,
    verbose=2,
    scoring='f1_macro',
    return_train_score=True
)
print("Starting grid search...")
grid_search.fit(X_train, y_train)
print("\n" + "="*50)
print("GRID SEARCH RESULTS")
print("="*50)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Best estimator: {grid_search.best_estimator_}")

Starting grid search...
Fitting 3 folds for each of 5 candidates, totalling 15 fits

GRID SEARCH RESULTS
Best parameters: {'C': 0.02, 'class_weight': 'balanced', 'loss': 'squared_hinge', 'max_iter': 20000, 'penalty': 'l2'}
Best CV score: 0.4862
Best estimator: LinearSVC(C=0.02, class_weight='balanced', dual=True, max_iter=20000,
          random_state=42)


In [21]:
# Train the final model with the best parameters
best_params = grid_search.best_params_
svm_model = LinearSVC(**best_params, random_state=42, dual=dual_choice)
svm_model.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,True
,tol,0.0001
,C,0.02
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


In [22]:
from sklearn.metrics import classification_report

# Make predictions
y_pred = svm_model.predict(X_test)

# Print detailed results
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0, target_names=emotions, digits=3))

Classification Report:
              precision    recall  f1-score   support

       anger      0.782     0.551     0.647        78
     disgust      0.000     0.000     0.000        24
        fear      0.733     0.367     0.489        30
         joy      0.821     0.365     0.505       126
     sadness      0.417     0.128     0.196        39
    surprise      0.240     0.128     0.167        47
     neutral      0.354     0.864     0.502       125

    accuracy                          0.467       469
   macro avg      0.478     0.343     0.358       469
weighted avg      0.551     0.467     0.442       469

